In [2]:
import sys
sys.path.append("..")

from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset
from src.utils import compute_metrics, set_global_seed
from src.data import load_and_preprocess_dataset


In [3]:
set_global_seed(42)

model_name = "distilbert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

tokenizer = AutoTokenizer.from_pretrained(model_name)

dataset = load_and_preprocess_dataset(tokenizer, max_length=128)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
total_params_before = sum(p.numel() for p in model.parameters())
trainable_params_before = sum(p.numel() for p in model.parameters() if p.requires_grad)

In [5]:
total_params_before, trainable_params_before

(66955010, 66955010)

In [6]:
for name, module in model.named_modules():
    if "linear" in str(type(module)).lower():
        print(name, type(module))

distilbert.transformer.layer.0.attention.q_lin <class 'torch.nn.modules.linear.Linear'>
distilbert.transformer.layer.0.attention.k_lin <class 'torch.nn.modules.linear.Linear'>
distilbert.transformer.layer.0.attention.v_lin <class 'torch.nn.modules.linear.Linear'>
distilbert.transformer.layer.0.attention.out_lin <class 'torch.nn.modules.linear.Linear'>
distilbert.transformer.layer.0.ffn.lin1 <class 'torch.nn.modules.linear.Linear'>
distilbert.transformer.layer.0.ffn.lin2 <class 'torch.nn.modules.linear.Linear'>
distilbert.transformer.layer.1.attention.q_lin <class 'torch.nn.modules.linear.Linear'>
distilbert.transformer.layer.1.attention.k_lin <class 'torch.nn.modules.linear.Linear'>
distilbert.transformer.layer.1.attention.v_lin <class 'torch.nn.modules.linear.Linear'>
distilbert.transformer.layer.1.attention.out_lin <class 'torch.nn.modules.linear.Linear'>
distilbert.transformer.layer.1.ffn.lin1 <class 'torch.nn.modules.linear.Linear'>
distilbert.transformer.layer.1.ffn.lin2 <class 't

In [7]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_lin", "k_lin", "v_lin", "out_lin", "lin1", "lin2"],
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    lora_dropout=0.1
)

In [8]:
model_lora = get_peft_model(model, lora_config)

In [9]:
model_lora.print_trainable_parameters()

trainable params: 1,255,682 || all params: 68,210,692 || trainable%: 1.8409


In [11]:
training_args = TrainingArguments(
    output_dir="./results/lora_sst2",
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=32,   
    num_train_epochs=5,              
    learning_rate=5e-4,              
    weight_decay=0.01,               
    warmup_steps=500,                
    lr_scheduler_type="cosine",      
    eval_strategy="epoch",     
    save_strategy="epoch",           
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=True,                       
    logging_dir="./logs",
    logging_steps=20,
    report_to=None,
)


In [ ]:
trainer = Trainer(
    model=model_lora,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

C:\Users\shiva\AppData\Local\Temp\ipykernel_25284\2344615087.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: shivamsinghml (shivamsingh-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,0.265200,0.286513,0.891055
2,0.241300,0.370344,0.884174
